# Memory-Efficient Linopy Usage: Best Practices

This notebook documents memory-efficient patterns for large-scale linopy models,
based on production experience with a 200-bus, 1800-generator PyPSA power system
model solved over 8760 hourly snapshots.

**Model scale (reference):**
- ~1.4M LP variables per weekly chunk
- ~1.4M LP constraints per weekly chunk  
- ~52 weekly chunks per year
- Steady-state memory: ~9 GB; peak during assembly: ~15 GB

## Contents
1. [Understanding linopy's memory layout](#1-understanding-linopys-memory-layout)
2. [Chunked time-horizon solving](#2-chunked-time-horizon-solving)
3. [LP blueprint reuse across chunks](#3-lp-blueprint-reuse-across-chunks)
4. [Efficient RHS and objective updates](#4-efficient-rhs-and-objective-updates)
5. [Memory profiling](#5-memory-profiling)
6. [Explicit cleanup between solves](#6-explicit-cleanup-between-solves)
7. [Sparse-friendly constraint formulation](#7-sparse-friendly-constraint-formulation)

## 1. Understanding linopy's memory layout

Before optimising, it helps to understand where memory actually lives in a linopy model.

### Data structures

A `linopy.Model` stores its data primarily in `xarray.Dataset` objects — one per
variable and one per constraint. For a model with $N$ variables and $M$ constraints
over $T$ timesteps:

| Structure | Shape | Approx. memory (float64) |
|-----------|-------|---------------------------|
| Variable labels (`.labels`) | $(T, N)$ | $8TN$ bytes |
| Variable bounds (`.lower`, `.upper`) | $2 \times (T, N)$ | $16TN$ bytes |
| Constraint labels + RHS + sign | $3 \times (T, M)$ | $24TM$ bytes |
| Expression term data (`_term` dim) | $(T, M, k)$ where $k$ = avg. terms | variable |
| Sparse matrix $A$ (CSC) | $\text{nnz} \times 3$ arrays | $24 \cdot \text{nnz}$ bytes |
| Solution + dual vectors | $(N + M)$ | $8(N+M)$ bytes |

For a 1.4M variable / 1.4M constraint model with $T=168$ snapshots:
- Variable storage: ~3.4 GB  
- Constraint storage: ~1.2 GB  
- Sparse $A$ matrix (~50M non-zeros): ~1.2 GB  
- **Peak during `matrices.A` assembly: ~3× the matrix size (~3.6 GB spike)**
- **Peak during solution unpacking: ~3× variable count** (solver array + pandas indexed copy + xarray result)

### The two memory spikes

There are two distinct memory peaks in a linopy solve cycle:

**1. `constraints.flat` (triggered by `matrices.A`)**: Before building the sparse
matrix, linopy concatenates all constraints into a pandas DataFrame with ~7 columns
and $(M \times T \times k)$ rows (where $k$ = average terms per constraint, typically
2–4). At 120 bytes/row of pandas overhead, this temporarily consumes several GB
*before* the CSC matrix itself exists.

**2. COO→CSC conversion**: The flat DataFrame is then converted to COO format
(row, col, data arrays) then to CSC. The peak is roughly 3× the final matrix size —
both the COO intermediate and the final CSC exist in memory simultaneously during
conversion.

**3. Solution unpacking**: After solve, the dense solver solution vector is unpacked
into xarray DataArrays per variable. Three copies exist briefly: solver numpy array
→ pandas Series with int index → xarray DataArray. This adds ~3× N_vars × 8 bytes
of temporary memory.

Avoid triggering repeated matrix builds — they repeat spikes 1 and 2 each time:

In [ ]:
import numpy as np

# BAD: triggers matrix build twice
A = model.matrices.A  # builds CSC matrix
b = model.matrices.b  # already cached — OK
A2 = model.matrices.A  # re-builds if cache was cleared!

# GOOD: access once, store locally
matrices = model.matrices
A = matrices.A
b = matrices.b
c = matrices.c

### Checking sparsity

The sparse $A$ matrix only saves memory if the model is actually sparse. Check before
committing to large-scale solving:

In [ ]:
A = model.matrices.A
n_rows, n_cols = A.shape
density = A.nnz / (n_rows * n_cols)
print(f"Matrix shape: {n_rows:,} × {n_cols:,}")
print(f"Non-zeros: {A.nnz:,}")
print(f"Density: {density:.4%}")
print(f"Memory (CSC): {A.data.nbytes / 1e9:.2f} GB")
print(f"Dense equivalent: {n_rows * n_cols * 8 / 1e9:.1f} GB")
# Typical power systems: <0.01% density → >99% savings

## 2. Chunked time-horizon solving

For year-long horizons (8760 h), building a single linopy model is impractical —
it would require ~500 GB for the LP alone. The standard approach is **chunked solving**:
solve weekly (168 h) sub-problems sequentially, passing boundary conditions between
chunks (e.g., storage state-of-charge).

### Basic chunking pattern

In [ ]:
import ctypes
import gc
import platform

import pypsa

CHUNK_HOURS = 168  # 1 week


def release_memory():
    """Return freed pages to the OS (important on Linux/macOS)."""
    gc.collect()
    if platform.system() == "Darwin":
        libc = ctypes.CDLL("libSystem.B.dylib")
        libc.malloc_zone_pressure_relief(0, 0)
    elif platform.system() == "Linux":
        libc = ctypes.CDLL("libc.so.6")
        libc.malloc_trim(0)


def solve_chunked(network: pypsa.Network, chunk_hours: int = CHUNK_HOURS):
    snapshots = network.snapshots
    n_chunks = int(np.ceil(len(snapshots) / chunk_hours))
    results = []

    for i in range(n_chunks):
        chunk_snaps = snapshots[i * chunk_hours : (i + 1) * chunk_hours]
        print(f"Chunk {i + 1}/{n_chunks}: {chunk_snaps[0]} – {chunk_snaps[-1]}")

        status, condition = network.optimize(
            solver_name="highs",
            snapshots=chunk_snaps,
            io_api="direct",  # avoids writing LP files to disk
        )

        if status == "ok":
            results.append(network.generators_t.p[chunk_snaps].copy())

        # Critical: free the linopy model before the next chunk
        if hasattr(network, "model"):
            del network.model
        release_memory()

    return results

### Why `del network.model` matters

After `network.optimize()`, the linopy model (including the xarray Datasets, sparse
matrix, and solver model) is attached as `network.model`. It is **not** freed until
you delete it explicitly — Python's reference count keeps it alive.

At 1.4M variables/constraints over 168 snapshots, `network.model` consumes ~9 GB.
Without `del`, each of 52 chunks accumulates: after 2 chunks you have 18 GB resident.

The `release_memory()` call after `del` is equally important on Linux: Python's
allocator (`malloc`) does not return freed heap pages to the OS until
`malloc_trim(0)` is called. Without it, RSS stays high even though Python
considers memory free.

## 3. LP blueprint reuse across chunks

For models where the **LP structure** (variables, constraints, sparsity pattern) is
identical across chunks and only **right-hand-side** values and **objective
coefficients** change, you can skip the linopy rebuild entirely for chunks 2+.

The pattern:
1. **Chunk 1**: Build the full linopy model, solve, extract the HiGHS object and
   enough metadata to update it directly.
2. **Chunks 2+**: Call `changeRowsBounds()` / `changeColsCost()` on the cached
   HiGHS object, then re-solve — no linopy involved.

This saves ~50 s and ~9 GB of allocation per chunk.

In [ ]:
from dataclasses import dataclass, field
from typing import Any


@dataclass
class LPBlueprint:
    """Cached LP structure from chunk 1, reused for chunks 2+."""

    highs: Any  # highspy.Highs object
    n_rows: int
    n_cols: int
    sense: np.ndarray  # '<' / '>' / '=' per row
    tv_row_indices: np.ndarray  # rows that change between chunks
    tv_col_indices: np.ndarray  # objective columns that change
    # maps variable/constraint names → HiGHS column/row index arrays
    var_keys: dict = field(default_factory=dict)
    con_keys: dict = field(default_factory=dict)


def extract_blueprint(model) -> LPBlueprint:
    """
    Extract LP blueprint from a solved linopy model (chunk 1).

    Call immediately after network.optimize() while network.model is still alive.
    """
    h = model.solver_model  # highspy.Highs
    n_rows = h.getNumRow()
    n_cols = h.getNumCol()

    # Identify time-varying rows: those whose RHS changes each chunk.
    # In PyPSA this is generator p_max/p_min bounds + power balance rows.
    # Tag them during model build, or detect by name pattern.
    tv_rows = _identify_tv_rows(model)  # implementation-specific
    tv_cols = _identify_tv_objective_cols(model)  # implementation-specific

    # Extract constraint sense once (does not change between chunks)
    sense = np.array([h.getRowSense(r) for r in range(n_rows)])

    bp = LPBlueprint(
        highs=h,
        n_rows=n_rows,
        n_cols=n_cols,
        sense=sense,
        tv_row_indices=tv_rows,
        tv_col_indices=tv_cols,
    )

    # Cache variable → HiGHS column index mappings
    for var_name, var in model.variables.items():
        bp.var_keys[var_name] = var.labels.values.copy()  # shape (T, N_comp)

    # Detach HiGHS from linopy so it survives del network.model
    model.solver_model = None
    return bp


def reuse_blueprint(bp: LPBlueprint, new_rhs: np.ndarray, new_obj: np.ndarray):
    """
    Update and re-solve using cached HiGHS object. Skips linopy entirely.

    Parameters
    ----------
    bp       : blueprint from extract_blueprint()
    new_rhs  : updated RHS values for time-varying rows (len = len(bp.tv_row_indices))
    new_obj  : updated objective coefficients for time-varying columns
    """
    h = bp.highs

    # Update row bounds (only the ~15% that change)
    lower = np.where(bp.sense[bp.tv_row_indices] != "<", new_rhs, -np.inf)
    upper = np.where(bp.sense[bp.tv_row_indices] != ">", new_rhs, np.inf)
    h.changeRowsBounds(
        len(bp.tv_row_indices),
        bp.tv_row_indices.astype(np.int32),
        lower.astype(np.float64),
        upper.astype(np.float64),
    )

    # Update objective
    h.changeColsCost(
        len(bp.tv_col_indices),
        bp.tv_col_indices.astype(np.int32),
        new_obj.astype(np.float64),
    )

    # IMPORTANT: clear previous solution so presolve runs on the new data
    h.clearSolver()

    # Solve with same options as chunk 1
    h.run()

    return h.getInfoValue("primal_solution_status")[1]

### Why `clearSolver()` is essential

`h.clearSolver()` resets HiGHS's internal solver state (basis, solution vectors,
presolve reductions) without touching the LP data (rows, columns, matrix).

Without it, HiGHS may try to warm-start from a stale basis that corresponds to a
**different** RHS — leading to wrong solutions or spurious infeasibility.

After `clearSolver()`, presolve runs on the fresh data. For a typical power-systems
LP, presolve reduces 1.4M rows → 128K rows (~89% reduction), which is why chunks 2+
can be **faster** than chunk 1 even though they skip the linopy build overhead.

### Warm-starting across chunks (experimental)

An alternative to cold-start IPM on every chunk is:
- **Chunk 1**: solve with IPM + crossover to produce a simplex basis
- **Chunks 2+**: solve with dual simplex warm-started from the prior basis

This is most beneficial when consecutive chunks are structurally similar (e.g.,
adjacent weeks with similar weather). HiGHS maintainer @jajhall confirmed that
presolve correctly maps the basis through LP reductions, so warm-start works even
with presolve enabled (see [HiGHS issue #2911](https://github.com/ERGO-Code/HiGHS/issues/2911)).

```python
# Chunk 1: IPM + crossover
h.setOptionValue('solver', 'ipm')
h.setOptionValue('run_crossover', 'on')   # produces simplex basis
h.run()

# Chunks 2+: warm dual simplex (do NOT call clearSolver())
update_rhs_and_obj(h, new_rhs, new_obj)   # update data only
h.setOptionValue('solver', 'simplex')
h.setOptionValue('simplex_strategy', 1)   # dual simplex
h.run()
```

Whether warm simplex is faster than cold IPM depends on how much the basis changes
between chunks. For highly variable renewables (adjacent weeks differ greatly),
cold IPM may be competitive.

## 4. Efficient RHS and objective updates

When computing new RHS values from time-series data (e.g., hourly wind profiles),
vectorised numpy operations are significantly faster and more memory-efficient than
pandas operations.

In [ ]:
import numpy as np

# --- Generator upper bound RHS ---


# BAD: pandas path — allocates intermediate Series/DataFrame at every step
def compute_gen_upper_bad(network, snapshots, gen_names):
    p_max_pu = network.generators_t.p_max_pu.reindex(
        index=snapshots, columns=gen_names
    ).fillna(
        network.generators.p_max_pu[gen_names]
    )  # fillna broadcasts a Series → allocates a full (T × N) DataFrame
    return (p_max_pu * network.generators.p_nom[gen_names]).values


# GOOD: numpy path — broadcast static values without expanding
def compute_gen_upper_good(network, snapshots, gen_names):
    p_nom = network.generators.p_nom[gen_names].values  # (N,)
    p_max_static = network.generators.p_max_pu[gen_names].values  # (N,)

    if len(network.generators_t.p_max_pu.columns) > 0:
        p_max_pu = network.generators_t.p_max_pu.reindex(
            index=snapshots, columns=gen_names
        ).values  # convert to numpy immediately — (T × N) float64

        # Fill NaNs via broadcast — avoids allocating a second (T × N) array
        nan_mask = np.isnan(p_max_pu)
        if nan_mask.any():
            # np.broadcast_to returns a read-only view, not a copy
            static_broadcast = np.broadcast_to(
                p_max_static[np.newaxis, :], p_max_pu.shape
            )
            p_max_pu[nan_mask] = static_broadcast[nan_mask]
    else:
        # All generators are static — use a view, not a copy
        p_max_pu = np.broadcast_to(
            p_max_static[np.newaxis, :], (len(snapshots), len(gen_names))
        )

    return p_max_pu * p_nom[np.newaxis, :]  # (T × N) — element-wise, in-place

### Load aggregation pattern

Aggregating time-varying loads to buses requires a groupby. The pandas transpose trick
avoids creating an intermediate transposed copy:

In [ ]:
def load_by_bus(network, snapshots, bus_names):
    """Aggregate per-load time series to per-bus totals."""
    loads = network.loads
    load_ts = network.loads_t.p_set.reindex(snapshots)

    # Group by bus using the transposition trick:
    # .T groups rows (loads) → .sum().T gives (snapshots × buses)
    grouped = load_ts.T.groupby(loads.bus).sum().T

    # Add static loads (p_set in generators, not generators_t)
    static_loads = loads[~loads.index.isin(load_ts.columns)]
    if len(static_loads) > 0:
        static_by_bus = static_loads.groupby("bus")["p_set"].sum()
        grouped = grouped.add(static_by_bus, fill_value=0.0)

    return grouped.reindex(columns=bus_names, fill_value=0.0)

### Sparse RHS assignment

When the constraint label array maps variable indices to HiGHS row/column indices,
some entries are `-1` (missing / not applicable). Always filter before writing:

In [ ]:
def push_rhs_to_highs(h, b_full, keys, sense):
    """
    Push a dense RHS array to HiGHS, skipping invalid entries.

    Parameters
    ----------
    h       : highspy.Highs object
    b_full  : (T × N) numpy array of RHS values (may contain NaN for unused entries)
    keys    : (T × N) int32 array of HiGHS row indices (-1 = missing)
    sense   : (n_rows,) array of '<', '>', '=' per row
    """
    flat_keys = keys.ravel()
    flat_b = b_full.ravel()
    valid = flat_keys >= 0

    idx = flat_keys[valid].astype(np.int32)
    vals = flat_b[valid].astype(np.float64)

    row_sense = sense[idx]
    lower = np.where(row_sense != "<", vals, -np.inf)
    upper = np.where(row_sense != ">", vals, np.inf)

    h.changeRowsBounds(len(idx), idx, lower, upper)

## 5. Memory profiling

Use `tracemalloc` (stdlib) or `memory_profiler` to identify where memory peaks occur.

In [ ]:
import tracemalloc


def profile_model_build(network, snapshots):
    """Profile memory usage during linopy model build and solve."""
    tracemalloc.start()

    # Snapshot 1: before model build
    snap0 = tracemalloc.take_snapshot()

    status, condition = network.optimize(
        solver_name="highs",
        snapshots=snapshots,
        io_api="direct",
    )

    # Snapshot 2: after model build + solve
    snap1 = tracemalloc.take_snapshot()

    # Report top allocations
    stats = snap1.compare_to(snap0, "lineno")
    print("Top 10 memory consumers:")
    for stat in stats[:10]:
        print(f"  {stat}")

    current, peak = tracemalloc.get_traced_memory()
    print(f"\nCurrent: {current / 1e9:.2f} GB")
    print(f"Peak:    {peak / 1e9:.2f} GB")

    tracemalloc.stop()
    return network.model

In [ ]:
# Quick memory estimate without solving
def estimate_model_memory(
    n_vars: int, n_cons: int, n_snapshots: int, avg_terms: float = 3.0
) -> dict:
    """
    Estimate peak memory for a linopy model.

    Returns dict with per-component estimates in GB.
    """
    N, M, T = n_vars, n_cons, n_snapshots
    nnz = int(M * T * avg_terms)  # approximate non-zeros in A

    return {
        "variable_labels": 3 * 8 * N * T / 1e9,  # labels + lower + upper
        "constraint_labels": 3 * 8 * M * T / 1e9,  # labels + rhs + sign
        "sparse_A_steady": 3 * 8 * nnz / 1e9,  # COO: data + row + col
        "sparse_A_peak": 3 * 3 * 8 * nnz / 1e9,  # 3× during assembly
        "solution_vectors": 8 * (N + M) * T / 1e9,
        "xarray_overhead_est": 0.1,  # rough estimate
    }


# Example: 1.4M vars, 1.4M constraints, 168 snapshots
est = estimate_model_memory(1_400_000, 1_400_000, 168)
total = sum(est.values())
for k, v in est.items():
    print(f"  {k:<30} {v:.2f} GB")
print(f"  {'TOTAL':<30} {total:.2f} GB")

## 6. Explicit cleanup between solves

Linopy caches several computed properties on the `matrices` accessor. These are
invalidated automatically when the model changes, but if you hold references or
are debugging, they can prevent garbage collection.

### Which properties are cached vs recomputed

On `model.matrices`, properties split into two groups:

**`@cached_property` — stored in `__dict__`, freed by `clean_cached_properties()`:**
- `flat_vars` — pandas DataFrame of all variables (large)
- `flat_cons` — pandas DataFrame of all constraints (large, ~7 columns × M×T×k rows)
- `sol` — dense solution value vector
- `dual` — dense dual value vector

**`@property` — recomputed on every access (no cache to clear):**
- `A`, `b`, `c`, `sense`, `lb`, `ub`, `vlabels`, `clabels` — matrix/vector accessors

The `flat_cons` DataFrame is typically the biggest cached object (often 2–4 GB). It
is created by `matrices.A` and kept in `__dict__` until explicitly cleared or the
model is deleted. `clean_cached_properties()` removes all `@cached_property` entries
from `__dict__` in one call.

### Cache invalidation

In [ ]:
# After solving chunk 1, before building chunk 2:

# 1. Explicitly clear cached matrix properties
if hasattr(network, "model") and hasattr(network.model, "matrices"):
    network.model.matrices.clean_cached_properties()  # frees ~2-3 GB

# 2. Detach the solver model before deleting (prevents double-free)
if hasattr(network, "model") and hasattr(network.model, "solver_model"):
    network.model.solver_model = None

# 3. Delete the linopy model
if hasattr(network, "model"):
    del network.model

# 4. Force Python GC + return pages to OS
gc.collect()
# Linux:
# ctypes.CDLL('libc.so.6').malloc_trim(0)
# macOS:
# ctypes.CDLL('libSystem.B.dylib').malloc_zone_pressure_relief(0, 0)

### Monitoring RSS during a run

In [ ]:
import os

import psutil


def rss_gb() -> float:
    """Return current process RSS in GB."""
    return psutil.Process(os.getpid()).memory_info().rss / 1e9


def solve_chunked_monitored(network, chunk_hours=168):
    snapshots = network.snapshots
    n_chunks = int(np.ceil(len(snapshots) / chunk_hours))

    for i in range(n_chunks):
        chunk_snaps = snapshots[i * chunk_hours : (i + 1) * chunk_hours]
        rss_before = rss_gb()

        network.optimize(
            solver_name="highs",
            snapshots=chunk_snaps,
            io_api="direct",
        )
        rss_during = rss_gb()

        if hasattr(network, "model"):
            del network.model
        release_memory()
        rss_after = rss_gb()

        print(
            f"Chunk {i + 1:2d}/{n_chunks} | "
            f"before={rss_before:.1f} GB  "
            f"during={rss_during:.1f} GB  "
            f"after={rss_after:.1f} GB  "
            f"freed={rss_during - rss_after:.1f} GB"
        )

## 7. Sparse-friendly constraint formulation

The sparsity pattern of the $A$ matrix depends heavily on how constraints are
formulated. A few linopy-specific tips:

### Use `merge=False` when constraints are per-snapshot

When constraints only involve variables from the **same** snapshot (e.g., nodal
power balance), pass `merge=False` to avoid linopy trying to align coordinates
across snapshot dimensions:

In [ ]:
# Nodal power balance: sum of generator output = load at each bus per snapshot
# Both sides are (snapshot × bus) aligned — no cross-snapshot terms
model.add_constraints(
    gen_power.groupby("bus").sum() == load_by_bus,
    name="power_balance",
    # merge=False avoids unnecessary coordinate broadcasting
)

### Avoid dense broadcasting in constraint expressions

Multiplying a variable by a full $(T \times N)$ coefficient array is fine.
But creating an expression involving **all snapshots × all generators × all buses**
simultaneously will produce a dense term tensor that blows up memory:

In [ ]:
# BAD: creates a (T × N_gen × N_bus) term array
# incidence_matrix is (N_gen × N_bus) → broadcasting creates full product
gen_at_bus = gen_vars * incidence_matrix  # AVOID for large N_gen, N_bus

# GOOD: group by bus assignment using pandas/xarray groupby
# This processes one bus at a time without materialising the full product
gen_power_by_bus = gen_vars.groupby("bus").sum()  # (T × N_bus) — sparse aggregation

### Use `filter_missings=True` (default) in `to_file` / `to_matrix`

The default `filter_missings=True` removes rows/columns with all-NaN labels before
building the sparse matrix. For models with many generators that are inactive for
part of the horizon (e.g., solar PV at night), this can halve the matrix size.

## Summary: quick-reference checklist

| Pattern | Savings | When to use |
|---------|---------|-------------|
| `del network.model` + `release_memory()` after each chunk | ~9 GB per chunk | Always |
| `io_api='direct'` | Avoids LP file I/O | Always for HiGHS |
| LP blueprint reuse (skip linopy for chunks 2+) | ~50 s + ~9 GB per chunk | When LP structure is identical across chunks |
| `clearSolver()` before re-solve | Correct presolve on new data | With blueprint reuse |
| Numpy broadcast fill (not `fillna`) | Avoids intermediate (T×N) copy | RHS/objective computation |
| Sparse RHS push (`valid = keys >= 0`) | Smaller HiGHS API call | Always with label arrays |
| `filter_missings=True` | Smaller $A$ matrix | When generators have gaps |
| `matrices.clean_cached_properties()` | ~2–3 GB after solve | Before `del network.model` |
| `tracemalloc` / `psutil.rss_gb()` | Identify peak locations | Profiling phase |